# Bunuelo - Esquematico (interactivo)

Capsula 9. Aqui **no simulamos todavia** (eso es Arroz con Leche). Exploramos
el **netlist** del OTA de 5T que dibujamos en xschem: que transistor conecta
con que nodo, sus tamanos, y la **fisica de primer orden** que guia por que el
disenador eligio esos W/L.

> Los numeros de corriente/ganancia de abajo son **estimaciones didacticas de
> primer orden** (ley cuadratica). Los valores reales, con los modelos de
> SKY130, salen en la siguiente capsula con SPICE.

Corre las celdas en orden.

## 1 - El netlist del OTA (lo que genero xschem)

In [ ]:
import re

# Netlist REAL del OTA (common/analog/ota5t/OTA_5T.spice), embebido:
NETLIST = '''
.subckt OTA_5T VIN VOUT_AMP VINN VINP IB VSS
XM1 IB IB VSS VSS sky130_fd_pr__nfet_01v8 L=1 W=2.5 nf=1 m=4
XM2 tail IB VSS VSS sky130_fd_pr__nfet_01v8 L=1 W=2.5 nf=1 m=4
XM3 net1 VINP tail VSS sky130_fd_pr__nfet_01v8 L=1 W=3 nf=1 m=4
XM4 VOUT_AMP VINN tail VSS sky130_fd_pr__nfet_01v8 L=1 W=3 nf=1 m=4
XM5 VOUT_AMP net1 VIN VIN sky130_fd_pr__pfet_01v8 L=1 W=6 nf=1 m=4
XM6 net1 net1 VIN VIN sky130_fd_pr__pfet_01v8 L=1 W=6 nf=1 m=4
XM8 tail VSS tail VSS sky130_fd_pr__nfet_01v8 L=0.15 W=3 nf=1 m=4
XM7 VIN VIN VIN VIN sky130_fd_pr__pfet_01v8 L=0.15 W=6 nf=1 m=4
.ends
'''

def parse(netlist):
    fets = []
    for ln in netlist.strip().splitlines():
        t = ln.split()
        if not t or not t[0].upper().startswith('XM'):
            continue
        name, d, g, s, b, model = t[0], t[1], t[2], t[3], t[4], t[5]
        params = dict(p.split('=') for p in t[6:] if '=' in p)
        kind = 'nfet' if 'nfet' in model else 'pfet'
        fets.append(dict(name=name, kind=kind, D=d, G=g, S=s, B=b,
                         L=float(params.get('L',1)), W=float(params.get('W',1)),
                         m=int(float(params.get('m',1)))))
    return fets

fets = parse(NETLIST)
print(f'{len(fets)} transistores en el OTA')
for f in fets:
    print(f"  {f['name']:4} {f['kind']}  D={f['D']:9} G={f['G']:6} S={f['S']:5} "
          f"W/L={f['W']}/{f['L']}  m={f['m']}")

## 2 - Tabla de dispositivos

Cada transistor: su **tipo** (nfet/pfet), los **nodos** a los que va cada
terminal (Drain/Gate/Source/Bulk) y sus **tamanos** (W/L y el multiplicador m).
Esto es, literalmente, el circuito.

In [ ]:
import pandas as pd
df = pd.DataFrame(fets)[['name','kind','D','G','S','B','W','L','m']]
df = df.rename(columns={'name':'FET','kind':'tipo','W':'W(um)','L':'L(um)'})
df

## 3 - El grafo del circuito

Sin abrir xschem: dibujamos quien se conecta con quien. Los **nodos
electricos** (VIN, VSS, tail, net1, VOUT_AMP...) en ambar; los **transistores**
en azul (nfet) o rojo (pfet). Cada arista es un terminal (D/G/S).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
nodes_elec, nodes_fet = set(), []
for f in fets:
    G.add_node(f['name'], bip='fet', kind=f['kind']); nodes_fet.append(f['name'])
    for term in ('D','G','S'):
        n = f[term]
        G.add_node(n, bip='elec'); nodes_elec.add(n)
        G.add_edge(f['name'], n, term=term)

pos = nx.kamada_kawai_layout(G)
fig, ax = plt.subplots(figsize=(10, 7))
nx.draw_networkx_nodes(G, pos, nodelist=sorted(nodes_elec), node_color='#C8742B',
                       node_size=1500, node_shape='o', ax=ax)
nfet = [f['name'] for f in fets if f['kind']=='nfet']
pfet = [f['name'] for f in fets if f['kind']=='pfet']
nx.draw_networkx_nodes(G, pos, nodelist=nfet, node_color='#2563eb',
                       node_size=1100, node_shape='s', ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=pfet, node_color='#dc2626',
                       node_size=1100, node_shape='s', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#999', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_color='white',
                        font_weight='bold', ax=ax)
elabels = {(u,v): d['term'] for u,v,d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=elabels, font_size=7, ax=ax)
ax.set_title('OTA 5T - grafo de conexiones  (ambar=nodo, azul=nfet, rojo=pfet)')
ax.axis('off'); plt.tight_layout(); plt.show()

## 4 - Juega con el tamano: corriente, gm y ganancia

El disenador no describe *que* hace el circuito: elige **W/L** de cada
transistor. Mira como, en la region de saturacion (ley cuadratica de primer
orden), el tamano y el **overdrive** (Vov = Vgs - Vth) fijan la corriente, la
transconductancia **gm** y la **ganancia intrinseca** Av0 = gm*ro.

- **W mas ancho** -> mas corriente y mas gm (mas 'fuerza').
- **L mas largo** -> menos lambda -> mas ro -> mas **ganancia** (por eso el
  core usa L=1 um, no el minimo).
- Dato clave analogico: **Av0 no depende de W** -- solo del overdrive y L.

In [ ]:
import numpy as np
from ipywidgets import interact, FloatSlider, Dropdown

# Constantes de PRIMER ORDEN (didacticas, ~SKY130 1.8V). No son signoff.
KP   = {'nfet': 120e-6, 'pfet': 45e-6}   # u*Cox  [A/V^2]
VTH  = {'nfet': 0.45,  'pfet': 0.45}     # |Vth|  [V]
LAML = 0.04   # lambda*L  [1/V * um]  -> lambda = LAML / L

by_name = {f['name']: f for f in fets}

def explora(FET='XM3', W_um=3.0, L_um=1.0, Vov=0.2):
    f = by_name[FET]; kind = f['kind']; m = f['m']
    kp, vth = KP[kind], VTH[kind]
    lam = LAML / L_um
    WL = W_um / L_um
    vov = np.linspace(0.01, 0.6, 200)
    Id_curve = 0.5*kp*WL*vov**2 * m * 1e6            # uA (incluye m fingers)
    Id = 0.5*kp*WL*Vov**2 * m                        # A
    gm = kp*WL*Vov * m                               # S
    ro = 1.0/(lam*Id) if Id>0 else float('inf')      # ohm
    Av0 = gm*ro                                      # V/V (adimensional)
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.plot(vov, Id_curve, color='#C8742B', lw=2)
    ax.plot([Vov],[Id*1e6], 'o', color='#dc2626', ms=9)
    ax.set_xlabel('Vov = Vgs - Vth  [V]'); ax.set_ylabel('Id  [uA]')
    ax.set_title(f"{FET} ({kind}, m={m})  W/L={W_um}/{L_um}  |  "
                 f"Id={Id*1e6:.1f} uA   gm={gm*1e6:.0f} uS   Av0={Av0:.0f} V/V")
    ax.grid(alpha=0.3)
    ax.text(0.02, 0.95, 'ley cuadratica (1er orden)', transform=ax.transAxes,
            fontsize=8, color='#666', va='top')
    plt.tight_layout(); plt.show()

interact(explora,
         FET=Dropdown(options=list(by_name), value='XM3', description='FET'),
         W_um=FloatSlider(value=3.0, min=0.5, max=20, step=0.5, description='W (um)'),
         L_um=FloatSlider(value=1.0, min=0.15, max=4, step=0.05, description='L (um)'),
         Vov=FloatSlider(value=0.2, min=0.05, max=0.5, step=0.01, description='Vov (V)'));

## 5 - Lo que sigue

Tenemos el circuito **capturado** (grafo + tamanos) y la intuicion de por que
esos W/L. Pero esto fue **primer orden**: falta meterlo a SPICE con los modelos
reales de SKY130 y ver la ganancia, el ancho de banda y los **corners**.

🧅 Bunuelo &nbsp;->&nbsp; 🍚 Arroz con Leche (Simulacion)